# Pratica — Lezione 7. Deep Q-Networks (DQN)

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

In questa pratica lavori sullo stesso DQN (Deep Q-Network) e sullo stesso ambiente CartPole della classe. Due esercizi:

1. Completi tu stesso il calcolo della perdita di DQN, il pezzo che hai visto nel blocco tre della classe
2. Programmi un decadimento di epsilon e confronti la sua curva di ricompensa con quella dell'epsilon fisso usato in classe

**Prima di iniziare, esegui le celle seguenti per avere pronta la struttura di partenza.** Contengono l'installazione di `gymnasium`, le importazioni, gli iperparametri, la rete Q, il buffer di replay e la politica epsilon-greedy: sono identici a quelli della classe, quindi non serve modificarli.

In [ ]:
# Instalamos gymnasium: es la UNICA dependencia externa de esta practica.
# PyTorch, numpy y matplotlib ya vienen preinstalados en Google Colab.
!pip install gymnasium


In [ ]:
import random
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import gymnasium as gym

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

# --- Hiperparametros (identicos a los de la clase) ---
NOMBRE_ENTORNO = "CartPole-v1"
NUM_EPISODIOS = 200                    # episodios de entrenamiento
GAMMA = 0.99                           # factor de descuento
TASA_APRENDIZAJE = 1e-3
TAMANO_BUFFER = 10000                  # capacidad del buffer de repeticion
TAMANO_LOTE = 64                       # tamano del mini-lote muestreado en cada paso
PASOS_ACTUALIZAR_RED_OBJETIVO = 500    # cada cuantos pasos se sincroniza la red objetivo
PASOS_MINIMOS_ANTES_ENTRENAR = 1000    # esperamos a tener algo de experiencia antes de entrenar


In [ ]:
class RedQ(nn.Module):
    """Aproximador de funciones: recibe un estado y devuelve un valor
    estimado para cada accion posible (identica a la de la clase)."""

    def __init__(self, dim_estado, num_acciones):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(dim_estado, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_acciones),
        )

    def forward(self, estado):
        return self.capas(estado)


Transicion = namedtuple(
    "Transicion", ["estado", "accion", "recompensa", "siguiente_estado", "terminado"]
)


class BufferRepeticion:
    """Buffer de repeticion de experiencias, identico al de la clase."""

    def __init__(self, capacidad):
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, *args):
        self.memoria.append(Transicion(*args))

    def muestrear(self, tamano_lote):
        return random.sample(self.memoria, tamano_lote)

    def __len__(self):
        return len(self.memoria)


def elegir_accion(red_online, estado, epsilon, num_acciones):
    """Politica epsilon-greedy, identica a la de la clase."""
    if random.random() < epsilon:
        return random.randrange(num_acciones)
    with torch.no_grad():
        estado_tensor = torch.tensor(estado, dtype=torch.float32).unsqueeze(0)
        valores_q = red_online(estado_tensor)
        return int(torch.argmax(valores_q, dim=1).item())


entorno = gym.make(NOMBRE_ENTORNO)
DIM_ESTADO = entorno.observation_space.shape[0]
NUM_ACCIONES = entorno.action_space.n
entorno.close()

print("Dimension del estado:", DIM_ESTADO, "| Numero de acciones:", NUM_ACCIONES)


## Esercizio 1 — Completa la perdita di DQN da zero

In classe hai visto la formula della perdita di DQN:

`L(θ) = E[ (r + γ · max_a' Q(s',a';θ⁻) − Q(s,a;θ))² ]`

Qui sotto trovi `entrenar_paso` con la struttura di partenza già costruita: la preparazione del mini-batch in tensori e il calcolo di `valores_q` (la previsione, `Q(s,a;θ)`) sono già completi. **Il tuo compito è riempire i due `# TODO`:**

1. Calcola `objetivo`, il valore r + γ · max_a' Q(s',a';θ⁻), usando `red_objetivo` dentro un blocco `with torch.no_grad():` (lo "stop-gradient": l'errore non deve mai fluire verso la rete obiettivo)
2. Calcola `perdida`, l'errore quadratico medio (MSE) tra `valores_q` (la previsione) e `objetivo`

**Suggerimenti**

- `red_objetivo(siguientes_estados)` ti restituisce, per ogni transizione del lotto, il valore Q delle due azioni nello stato successivo; con `.max(dim=1)[0]` tieni il più grande dei due, cioè `max_a' Q(s',a';θ⁻)`
- Quando `terminado` vale uno (l'episodio è finito), non c'è uno stato successivo che valga la pena guardare: moltiplica il termine `GAMMA * max_q_siguiente` per `(1 - terminados)` per annullarlo in quel caso
- `nn.functional.mse_loss(prediccion, objetivo)` calcola l'errore quadratico medio tra due tensori

Se lasci i `# TODO` non completati, la cella di prova qui sotto genererà un errore: è il segnale che ti manca da terminare il calcolo.

In [ ]:
def entrenar_paso(red_online, red_objetivo, buffer, optimizador):
    """Un paso de entrenamiento de DQN: muestrea un mini-lote del buffer y
    ajusta la red online para acercar su prediccion al objetivo calculado
    con la red objetivo (target network)."""
    lote = buffer.muestrear(TAMANO_LOTE)
    estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
    acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64).unsqueeze(1)
    recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
    siguientes_estados = torch.tensor(
        np.array([t.siguiente_estado for t in lote]), dtype=torch.float32
    )
    terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

    # Q(s,a;theta): valor que la red online asigna a la accion que de verdad se tomo
    # (ya completo: no hace falta tocarlo)
    valores_q = red_online(estados).gather(1, acciones).squeeze(1)

    # TODO 1: calcula "objetivo" = r + gamma * max_a' Q(s',a';theta-), usando
    # red_objetivo dentro de un bloque `with torch.no_grad():`. Recuerda anular
    # el segundo termino cuando el episodio termino, multiplicando por (1 - terminados).
    # objetivo = ...

    # TODO 2: calcula "perdida" como el error cuadratico medio (MSE) entre
    # valores_q (la prediccion) y objetivo.
    # perdida = ...

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


**Esegui la cella seguente per addestrare con il tuo `entrenar_paso` e verificare se funziona.** Usa lo stesso epsilon fisso della classe (0,1). Se la tua implementazione è corretta, la ricompensa media degli ultimi 20 episodi dovrebbe salire chiaramente verso la fine dell'addestramento, proprio come hai visto in classe.

In [ ]:
EPSILON_EJ1 = 0.1  # mismo epsilon fijo que usaste en clase

random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

entorno = gym.make(NOMBRE_ENTORNO)
red_online = RedQ(DIM_ESTADO, NUM_ACCIONES)
red_objetivo = RedQ(DIM_ESTADO, NUM_ACCIONES)
red_objetivo.load_state_dict(red_online.state_dict())
red_objetivo.eval()

optimizador = optim.Adam(red_online.parameters(), lr=TASA_APRENDIZAJE)
buffer = BufferRepeticion(TAMANO_BUFFER)

recompensas_ej1 = []
pasos_totales = 0

for episodio in range(NUM_EPISODIOS):
    estado, _ = entorno.reset(seed=SEMILLA + episodio)
    recompensa_episodio = 0.0
    terminado = False
    truncado = False

    while not (terminado or truncado):
        accion = elegir_accion(red_online, estado, EPSILON_EJ1, NUM_ACCIONES)
        siguiente_estado, recompensa, terminado, truncado, _ = entorno.step(accion)

        buffer.guardar(estado, accion, recompensa, siguiente_estado, terminado)
        estado = siguiente_estado
        recompensa_episodio += recompensa
        pasos_totales += 1

        if len(buffer) >= PASOS_MINIMOS_ANTES_ENTRENAR:
            entrenar_paso(red_online, red_objetivo, buffer, optimizador)

        if pasos_totales % PASOS_ACTUALIZAR_RED_OBJETIVO == 0:
            red_objetivo.load_state_dict(red_online.state_dict())

    recompensas_ej1.append(recompensa_episodio)

    if (episodio + 1) % 20 == 0:
        print(
            f"Episodio {episodio + 1}/{NUM_EPISODIOS} "
            f"| recompensa media (ultimos 20)={np.mean(recompensas_ej1[-20:]):.1f}"
        )

entorno.close()

plt.figure(figsize=(10, 5))
plt.plot(recompensas_ej1, alpha=0.4, label="Recompensa por episodio")
media_movil = np.convolve(recompensas_ej1, np.ones(20) / 20, mode="valid")
plt.plot(range(19, len(recompensas_ej1)), media_movil, label="Media movil (20 episodios)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa total")
plt.title("Ejercicio 1: tu perdida de DQN, con epsilon fijo")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def entrenar_paso(red_online, red_objetivo, buffer, optimizador):
    """Un passo di addestramento di DQN: campiona un mini-batch dal buffer e
    regola la rete online per avvicinare la sua previsione all'obiettivo
    calcolato con la rete obiettivo (target network)."""
    lote = buffer.muestrear(TAMANO_LOTE)
    estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
    acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64).unsqueeze(1)
    recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
    siguientes_estados = torch.tensor(
        np.array([t.siguiente_estado for t in lote]), dtype=torch.float32
    )
    terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

    # Q(s,a;theta): valore che la rete online assegna all'azione effettivamente eseguita
    valores_q = red_online(estados).gather(1, acciones).squeeze(1)

    # Obiettivo: r + gamma * max_a' Q(s',a';theta-), calcolato con la rete obiettivo.
    # torch.no_grad() implementa lo "stop-gradient": l'errore si propaga
    # solo verso la rete online, mai verso la rete obiettivo.
    with torch.no_grad():
        max_q_siguiente = red_objetivo(siguientes_estados).max(dim=1)[0]
        objetivo = recompensas + GAMMA * max_q_siguiente * (1 - terminados)

    # Perdita: errore quadratico medio tra la previsione e l'obiettivo
    perdida = nn.functional.mse_loss(valores_q, objetivo)

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()
```

Il punto chiave è il `with torch.no_grad():` attorno al calcolo di `objetivo`: senza di esso, PyTorch propagherebbe il gradiente anche verso `red_objetivo`, e spingeresti l'obiettivo stesso ad avvicinarsi alla tua previsione, invece che il contrario.
</details>

## Esercizio 2 — Programma il decadimento di epsilon e confrontalo con l'epsilon fisso

In classe hai usato un epsilon **fisso** (0,1) per tutto l'addestramento, per rendere il codice dal vivo più facile da seguire. Il manuale raccomanda invece un epsilon che **decade**: inizia esplorando molto (epsilon vicino a uno) e finisce esplorando poco (epsilon vicino a 0,05), man mano che la rete impara.

**Perché tu possa andare avanti anche se l'esercizio uno non ti è riuscito, questa cella include una copia già completa e funzionante di `entrenar_paso`. Il tuo compito è un altro: completare la funzione `calcular_epsilon`, dove c'è `# TODO`, con la formula di decadimento lineare.

**Il tuo compito**

1. Calcola `progreso`, la frazione di decadimento già percorsa: l'episodio corrente diviso per `EPISODIOS_DECAIMIENTO`, senza superare mai uno
2. Calcola `epsilon` interpolando linearmente tra `EPSILON_INICIAL` e `EPSILON_FINAL`, in base a quel progresso

**Suggerimenti**

- `progreso = min(episodio / EPISODIOS_DECAIMIENTO, 1.0)`
- L'interpolazione lineare tra due valori `A` e `B` secondo una frazione `p` è `A + p * (B - A)`: quando `p` vale zero restituisce `A`, e quando `p` vale uno restituisce `B`
- Qui `A` è `EPSILON_INICIAL` e `B` è `EPSILON_FINAL`

In [ ]:
def entrenar_paso(red_online, red_objetivo, buffer, optimizador):
    """Copia completa del paso de entrenamiento de DQN (identica a la solucion
    del ejercicio 1), para que puedas avanzar con este ejercicio de todos modos."""
    lote = buffer.muestrear(TAMANO_LOTE)
    estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
    acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64).unsqueeze(1)
    recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
    siguientes_estados = torch.tensor(
        np.array([t.siguiente_estado for t in lote]), dtype=torch.float32
    )
    terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

    valores_q = red_online(estados).gather(1, acciones).squeeze(1)

    with torch.no_grad():
        max_q_siguiente = red_objetivo(siguientes_estados).max(dim=1)[0]
        objetivo = recompensas + GAMMA * max_q_siguiente * (1 - terminados)

    perdida = nn.functional.mse_loss(valores_q, objetivo)

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


EPSILON_INICIAL = 1.0
EPSILON_FINAL = 0.05
EPISODIOS_DECAIMIENTO = 120    # episodios en los que epsilon baja de inicial a final


def calcular_epsilon(episodio):
    """Decaimiento lineal de epsilon: empieza en EPSILON_INICIAL y llega a
    EPSILON_FINAL en el episodio EPISODIOS_DECAIMIENTO, sin bajar mas alla."""
    # TODO 1: calcula "progreso", la fraccion del decaimiento ya recorrida
    # progreso = ...

    # TODO 2: interpola linealmente entre EPSILON_INICIAL y EPSILON_FINAL
    # epsilon = ...

    return epsilon


**Esegui la cella seguente.** Addestra due agenti con lo stesso `entrenar_paso` e gli stessi seed: uno con il tuo `calcular_epsilon` (decadimento) e uno con l'epsilon fisso della classe (0,1), per confrontare le due curve di ricompensa nello stesso grafico.

In [ ]:
def entrenar_dqn(funcion_epsilon):
    """Entrena un DQN completo y devuelve la recompensa por episodio.
    `funcion_epsilon(episodio)` decide el epsilon de cada episodio."""
    random.seed(SEMILLA)
    np.random.seed(SEMILLA)
    torch.manual_seed(SEMILLA)

    entorno = gym.make(NOMBRE_ENTORNO)
    red_online = RedQ(DIM_ESTADO, NUM_ACCIONES)
    red_objetivo = RedQ(DIM_ESTADO, NUM_ACCIONES)
    red_objetivo.load_state_dict(red_online.state_dict())
    red_objetivo.eval()

    optimizador = optim.Adam(red_online.parameters(), lr=TASA_APRENDIZAJE)
    buffer = BufferRepeticion(TAMANO_BUFFER)

    recompensas = []
    pasos_totales = 0

    for episodio in range(NUM_EPISODIOS):
        estado, _ = entorno.reset(seed=SEMILLA + episodio)
        recompensa_episodio = 0.0
        terminado = False
        truncado = False
        epsilon = funcion_epsilon(episodio)

        while not (terminado or truncado):
            accion = elegir_accion(red_online, estado, epsilon, NUM_ACCIONES)
            siguiente_estado, recompensa, terminado, truncado, _ = entorno.step(accion)

            buffer.guardar(estado, accion, recompensa, siguiente_estado, terminado)
            estado = siguiente_estado
            recompensa_episodio += recompensa
            pasos_totales += 1

            if len(buffer) >= PASOS_MINIMOS_ANTES_ENTRENAR:
                entrenar_paso(red_online, red_objetivo, buffer, optimizador)

            if pasos_totales % PASOS_ACTUALIZAR_RED_OBJETIVO == 0:
                red_objetivo.load_state_dict(red_online.state_dict())

        recompensas.append(recompensa_episodio)

    entorno.close()
    return recompensas


print("Entrenando con epsilon fijo (el de la clase)...")
recompensas_fijo = entrenar_dqn(lambda episodio: 0.1)

print("Entrenando con tu decaimiento de epsilon...")
recompensas_decay = entrenar_dqn(calcular_epsilon)

print("Recompensa media, ultimos 20 episodios, epsilon fijo:", np.mean(recompensas_fijo[-20:]))
print("Recompensa media, ultimos 20 episodios, epsilon con decaimiento:", np.mean(recompensas_decay[-20:]))


In [ ]:
def media_movil(valores, ventana=20):
    return np.convolve(valores, np.ones(ventana) / ventana, mode="valid")

plt.figure(figsize=(10, 5))
plt.plot(range(19, len(recompensas_fijo)), media_movil(recompensas_fijo), label="Epsilon fijo (0.1, el de la clase)")
plt.plot(range(19, len(recompensas_decay)), media_movil(recompensas_decay), label="Epsilon con decaimiento (el tuyo)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa total (media movil de 20 episodios)")
plt.title("Ejercicio 2: epsilon fijo vs. epsilon con decaimiento")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def calcular_epsilon(episodio):
    """Decadimento lineare di epsilon: parte da EPSILON_INICIAL e arriva a
    EPSILON_FINAL all'episodio EPISODIOS_DECAIMIENTO, senza scendere oltre."""
    # 1. Frazione del decadimento già percorsa, senza superare mai uno
    progreso = min(episodio / EPISODIOS_DECAIMIENTO, 1.0)

    # 2. Interpolazione lineare tra EPSILON_INICIAL e EPSILON_FINAL
    epsilon = EPSILON_INICIAL + progreso * (EPSILON_FINAL - EPSILON_INICIAL)

    return epsilon
```

Con questa formula, `epsilon` vale `EPSILON_INICIAL` all'episodio zero e scende in modo lineare fino a `EPSILON_FINAL` all'episodio `EPISODIOS_DECAIMIENTO`; da lì in poi, `progreso` resta fisso a uno e `epsilon` non scende più.

**Sul confronto**: non dare per scontato che una curva "vinca" sull'altra in modo definitivo con una sola ripetizione dell'esperimento. CartPole è un ambiente piccolo e il risultato dipende parecchio dal seed. L'importante è capire il meccanismo: un epsilon che decade esplora molto all'inizio, quando la rete non sa ancora nulla, e sfrutta di più in seguito, quando ha già imparato qualcosa di utile. Un epsilon fisso e basso, invece, esplora poco fin dal primo minuto, il che può ritardare il momento in cui l'agente scopre le traiettorie buone.
</details>

## Per continuare a esercitarti

Torna al notebook di classe e applica le idee di "Mettilo in pratica" sulla tua versione: **cambia `PASOS_ACTUALIZAR_RED_OBJETIVO`** tra 50, 500 e 5.000, oppure **cambia `TAMANO_BUFFER`** tra 500 e 50.000, e osserva come si trasforma la curva di ricompensa in ogni caso. Discutine con il tuo team: quale combinazione di iperparametri addestra in modo più stabile?